# Práctica: Modelos Probabilísticos en Recuperación de Información  
### Del modelo Two-Poisson a BM11 y BM25  

En esta práctica aprenderemos a:
- Entender el modelo probabilístico clásico (Binary Independence Model).
- Comprender el modelo Two-Poisson de Harter y su papel en la estimación de pesos.
- Implementar BM11 y BM25 desde cero.
- Ejecutar búsquedas en el corpus MEDLINE (1033 documentos).
- Analizar diferencias entre los modelos.

Duración estimada: **2 horas**

Cada celda incluye explicaciones para que puedas seguir el razonamiento paso a paso.

## 1. Carga del corpus Medline

In [ ]:
from pathlib import Path
import os

# Get the current directory
currentDirectory = Path().resolve()
# Go to P9/med
filePath = os.path.join(currentDirectory.parent, "P9", "med", "MED.ALL")

text = open(filePath, "r", encoding="utf8", errors="ignore").read()


def parse_medline_IW(text):
    docs = []
    for blk in text.split(".I ")[1:]:
        parts = blk.split(".W")
        if len(parts) > 1:
            docs.append(parts[1].strip().replace("\n", " "))
    return docs


docs = parse_medline_IW(text)
docs = [{"id": i, "text": d} for i, d in enumerate(docs)]
len(docs)

## 2. Tokenización y Preprocesamiento

Para aplicar cualquier modelo probabilístico necesitamos
representar cada documento mediante:
- su conjunto de tokens,
- su frecuencia de términos,
- su longitud.

Usaremos un preprocesamiento minimalista:  
minúsculas, eliminación de símbolos y split por espacios.

In [ ]:
import re
from collections import Counter


def tokenize(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    return [w for w in t.split() if w]


tokenized_docs = [tokenize(d["text"]) for d in docs]
doc_tf = [Counter(t) for t in tokenized_docs]
doc_len = [len(t) for t in tokenized_docs]
N = len(docs)

## 3. Modelo Probabilístico Clásico (Binary Independence Model)

Robertson y Sparck Jones demostraron que el peso óptimo del término en un
modelo probabilístico depende del **logaritmo de razones de probabilidad (odds ratio)**:

$$
w_t = \log \frac{p_t (1-u_t)}{u_t (1-p_t)}
$$

¿Qué representan $p_t$ y $u_t$?
* $p_t$: probabilidad de que el término $t$ aparezca en documentos relevantes.
* $u_t$: probabilidad de que el término aparezca en documentos no relevantes o en la colección general.

Estas probabilidades se estiman a partir de frecuencias de términos en conjuntos de documentos.

El peso $w_t$ se interpreta de la siguiente manera:

| Valor del peso | Significado |
|----------------|-------------|
| $w_t > 0$  | El término aparece más en documentos relevantes → **término útil** |
| $w_t < 0$  | El término aparece más en documentos no relevantes → **no es discriminante** |
| $w_t = 0$  | El término no aporta información para decidir |

Este es el **peso probabilístico óptimo**, pero en la práctica *no conocemos la relevancia de los documentos*, así que no podemos calcular $p_t$ ni $u_t$.


### Alternativa: Aproximación sin relevancia conocida (RSJ)

Robertson y Sparck Jones propusieron sustituir las probabilidades reales por **estimaciones basadas sólo en estadísticas de colección**, especialmente en la frecuencia de documento (df):

$$
u_t \approx \frac{df_t}{N}
$$

y $p_t$ se aproxima mediante un valor pequeño (derivado de suponer pocos relevantes), es decir:
$$
p_t \approx \frac{0.5}{N}
$$

Al sustituir estas aproximaciones y aplicar una corrección de suavizado ($+0.5$), el peso se convierte en:

$$
w_t \approx
\log
\frac{N - df_t + 0.5}{df_t + 0.5}
$$

donde:
- $df_t$: número de documentos que contienen el término,
- $N$: número total de documentos.


Esta es la fórmula **RSJ sin juicios de relevancia conocida**. Este peso es fundamental para Two-Poisson, BM11 y BM25.

### Interpretación

- Si un término aparece en **pocos documentos**, $df_t$ es pequeño y el peso es grande → término **discriminante**
- Si aparece en **muchos documentos**, el cociente se aproxima a 1 → peso pequeño → término **poco informativo**

Es decir:

$$
\text{RSJ} \approx \log \frac{N}{df_t}
$$

lo cual es muy parecido a la **IDF clásica**.



In [ ]:
df = Counter()
for tf in doc_tf:
    for w in tf:
        df[w] += 1

### ACTIVIDAD 1: Completar el código y comprobar que RSJ es similar a IDF para el término "glucose".

In [ ]:
import math


def idf(N, df, w):
    """
    IDF clásico: log(N / df)
    """
    return math.log(N / df[w])


def rsj(N, df, w):
    """
    RSJ aproximado sin relevancia conocida:
    log((N - df + 0.5) / (df + 0.5))
    """
    return math.log((N - df[w] + 0.5) / (df[w] + 0.5))


term = "glucose"

if term in df:
    df_t = df[term]
    print(f"Término: {term}")
    print(f"df = {df_t}")

    # Calcular IDF y RSJ
    print("IDF =", idf(N, df, term))
    print("RSJ =", rsj(N, df, term))

else:
    print(f"El término '{term}' no aparece en la colección.")

```
Término: glucose
df = 34
IDF = 3.413861944503477
RSJ = 3.366295829903141
```



## 4. Two-Poisson:

#### ¿Qué aporta?

El modelo **Two-Poisson de Harter** explica por qué ciertos términos
aparecen con diferente comportamiento en documentos relevantes y no relevantes:
1. **Apariciones accidentales**  
   En documentos **no relevantes**, los términos suelen aparecer muy poco.  
   Esto se modela con una **Poisson de media baja** (por ejemplo, λ₀ = 2).

2. **Apariciones sistemáticas**  
   En documentos **relevantes**, ciertos términos aparecen con más frecuencia  
   (por ejemplo, λ₁ = 6).  
   Esto se modela con una **Poisson de media alta**.

Este comportamiento se modela mediante **dos distribuciones Poisson**:

$$
P(tf_{t_i} = k) = (1 - \pi) \frac{e^{-\lambda_{0}} \lambda_{0}^{k}}{k!} +\;
\pi \frac{e^{-\lambda_{1}} \lambda_{1}^{k}}{k!}, \text{donde } \lambda_1 > \lambda_0
$$

- $k$: frecuencia del término  
- $\lambda_0$: media de la frecuencia de términos en no relevantes  
- $\lambda_1$: media de la frecuencia de términos en relevantes  
- $\pi$: proporción de documentos relevantes

Aunque no implementaremos las Poisson en sí, **Two-Poisson justifica**
el uso de pesos RSJ y la idea de que la frecuencia del término en el documento
debe saturar (no crecer linealmente para ranking).

### Explicación Visual

Mostramos dos distribuciones Poisson representando relevantes/no relevantes.

In [ ]:
import matplotlib.pyplot as plt
from math import exp, factorial
import numpy as np


def poisson_pmf(lmbda, k):
    return (lmbda**k * exp(-lmbda)) / factorial(k)


# Valores típicos
lambda_rel = (
    6  # supongamos que la media de la frecuencia de los términos en relevantes es 6
)
lambda_non = (
    2  # supongamos que la media de la frecuencia de los términos en no relevantes es 2
)
pi = 0.1  # proporción esperada de documentos relevantes

x = np.arange(0, 15)

# Distribuciones separadas
rel = [poisson_pmf(lambda_rel, k) for k in x]
non = [poisson_pmf(lambda_non, k) for k in x]

# Mezcla Two-Poisson
mix = [(1 - pi) * non[k] + pi * rel[k] for k in range(len(x))]

plt.plot(x, rel, label="Relevantes (λ=6)")
plt.plot(x, non, label="No relevantes (λ=2)")
plt.plot(x, mix, label=f"Mezcla Two-Poisson (π={pi})", linestyle="--")
plt.legend()
plt.title("Modelo Two-Poisson: Distribuciones de frecuencia")
plt.xlabel("Frecuencia k")
plt.ylabel("P(f=k)")
plt.show()

### ACTIVIDAD 2: Vamos a estimar los parámetros Two-Poisson para el término blood del corpus, es decir, tendremos que estimar:

- **λ₀**: media de frecuencias en documentos *no relevantes*
- **λ₁**: media de frecuencias en documentos *relevantes*
- **π**: proporción de documentos relevantes

Dado que no disponemos de juicios de relevancia, utilizaremos un método estadístico basado en la distribución de frecuencias del término.

---
**Pasos:**
 * 1. Calcular la frecuencia del término (tf) en cada documento : Obtén para cada documento cuántas veces aparece el término seleccionado.
 * 2. Considerar solo los documentos donde el término aparece: Filtra todos los documentos con $tf>0$
 * 3. Calcular el **percentil 80** de estas frecuencias. Utilizaremos este percentil como separación automática entre:
      - documentos donde el término aparece *ocasionalmente*  
      - documentos donde aparece *de forma intensa*  

      Interpretaremos:

      - **No relevantes** → documentos con `tf ≤ percentil80`
      - **Relevantes** → documentos con `tf > percentil80`
 * 4. Estimar los parámetros Two-Poisson como:
      - $\lambda_0 = \text{media de las frecuencias en documentos no relevantes}$
      - $\lambda_1 = \text{media de las frecuencias en documentos relevantes}$
      - $\pi = \frac{\text{número de documentos relevantes}}{\text{número total de documentos}}$
---

In [ ]:
# -------------------------------
# Frecuencia del término
# -------------------------------
term = "blood"  # patients
freqs = [doc.count(term) for doc in tokenized_docs]

print(f"Número de documentos donde aparece '{term}':", sum(f > 0 for f in freqs))

# -------------------------------
# Estimar λ₀, λ₁ y π
# -------------------------------

# Remove frequencies equal to zero
freqs = [f for f in freqs if f > 0]

# Calculate the cut with 80th percentile
cut = np.percentile(freqs, 80)

# Estimate λ₀, λ₁ and π
lambda_non = np.mean([f for f in freqs if f <= cut])
lambda_rel = np.mean([f for f in freqs if f > cut])
pi = len([f for f in freqs if f > cut]) / len(tokenized_docs)

print(f"cut = {cut:.1f}")
print(f"λ₀ (no relevantes) = {lambda_non:.4f}")
print(f"λ₁ (relevantes)   = {lambda_rel:.4f}")
print(f"π  (proporción relevantes) = {pi:.4f}")

#### **Preguntas:**

##### ¿El término parece relevante? (si λ₁ ≫ λ₀)

Sí.

##### ¿Es poco discriminante? (si λ₁ ≈ λ₀)

No, es discriminante.

##### ¿Es muy raro? (π muy pequeño)

Es muy pequeño, pero esto es porque el dataset es muy específico.

##### ¿Tiene un comportamiento bimodal?

No porque el dataset es muy específico. En un corpus más generalista, probablemente sí.

##### ¿Crees que este término sería útil para ranking en RI?

Sí, λ₁ es mucho mayor que λ₀.

In [ ]:
# -------------------------------
# Construir curvas
# -------------------------------
x = np.arange(0, 15)

non_curve = [poisson_pmf(lambda_non, k) for k in x]
rel_curve = [poisson_pmf(lambda_rel, k) for k in x]
mix_curve = [(1 - pi) * non_curve[k] + pi * rel_curve[k] for k in range(len(x))]

# -------------------------------
# Graficar
# -------------------------------
plt.figure(figsize=(8, 5))
plt.plot(x, non_curve, label=f"No relevantes (λ₀={lambda_non:.4f})")
plt.plot(x, rel_curve, label=f"Relevantes (λ₁={lambda_rel:.4f})")
plt.plot(x, mix_curve, "--", label=f"Mezcla Two-Poisson (π={pi:.4f})")

plt.title(f"Modelo Two-Poisson para el término '{term}'")
plt.xlabel("Frecuencia k")
plt.ylabel("P(f = k)")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Interpretación del gráfico

El gráfico muestra:

- La curva **Poisson relevante** .  
- La curva **Poisson no relevante**.  
- La **mezcla** de ambas distribuciones, ponderada por un parámetro π, que
  representa la proporción esperada de documentos relevantes en la colección.


### ¿Qué nos enseña el gráfico?

- Un término útil para RI aparece **muy poco** en no relevantes y **mucho más**
  en relevantes.
- La mezcla Two-Poisson refleja esta diferencia de comportamiento.
- En estos casos:
  * λ₀ → muy pequeño (el término NO aparece por accidente)
  * λ₁ → grande (el término aparece mucho cuando está presente)
  * π → pequeño pero no cero (una minoría de documentos son relevantes al término)

Este modelo es uno de los fundamentos teóricos más importantes de la RI clásica.

## 5. BM11: Primera simplificación práctica

El modelo Two-Poisson dice que la frecuencia de un término $t$ procede de dos comportamientos distintos:

* documentos no relevantes → Poisson con media $\lambda_0$
* documentos relevantes → Poisson con media $\lambda_1$

La clave:

* Si $\lambda_1$ ≫ $\lambda_0$, el término es discriminante

  → aparece muy poco por accidente
  → aparece mucho cuando es relevante

Esto sugiere que la frecuencia del término aporta evidencia de relevancia, pero no de forma lineal.

Pero Two-Poisson añade algo crucial: El modelo explica que la frecuencia del término en un documento debe saturar.

¿Qué significa “saturar”?

* A mayor frecuencia del término, más evidencia de relevancia, pero cada aparición adicional aporta menos evidencia que la anterior.

Esto se ve en la Two Poisson:

* en relevantes, la probabilidad sube con k, pero NO de manera lineal, sino con forma de Poisson.

Por tanto, el peso debe crecer como:

$$
\frac{tf_{t,d}}{k + tf_{t,d}}
$$

Por tanto, llegamos a BM11 que combina:

1. Pesos RSJ del Two Poisson.
2. Uso del tf del documento.
3. Saturación del tf.

La fórmula:

$$
BM_{11}(d,q)= \sum_{t \in q}
w_t \cdot \frac{tf_{t,d}}{tf_{t,d} + k}
$$

donde el parámetro $k$ controla la saturación del término y $w_t$ es
$$
w_t \approx
\log
\frac{N - df_t + 0.5}{df_t + 0.5}
$$

In [ ]:
def bm11(query, k=1.2, topk=10):
    q = tokenize(query)
    scores = []
    for i, tf in enumerate(doc_tf):
        s = 0
        for w in q:
            if w in tf:
                s += rsj(N, df, w) * (tf[w] / (tf[w] + k))
        scores.append((s, docs[i]["id"]))
    return sorted(scores, reverse=True)[:topk]

### Visualización saturación TF en BM11

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tf_vals = np.arange(1, 20)
k = 1.2
bm11_tf = tf_vals / (tf_vals + k)
plt.plot(tf_vals, bm11_tf)
plt.title("Saturación TF en BM11")
plt.xlabel("tf")
plt.ylabel("Contribución")
plt.show()

In [ ]:
term = "glucose"  # patients
freqs = np.array([doc.count(term) for doc in tokenized_docs])

# Filtrar tf>0 para ver sólo documentos donde aparece
tf_vals = np.sort(freqs[freqs > 0])

# ---- 3. Curva teórica BM11 ----
k = 1.2
bm11_curve = tf_vals / (tf_vals + k)

# ---- 4. Curva real empírica (normalizada) ----
# Normalizamos dividiendo por el máximo tf para visualizar
tf_real_norm = tf_vals / tf_vals.max()

# ---- 5. Graficar ----
plt.figure(figsize=(8, 5))
plt.plot(tf_vals, tf_real_norm, "o-", label=f"Saturación REAL del término '{term}'")
plt.plot(tf_vals, bm11_curve, "s--", label=f"Saturación teórica BM11", alpha=0.8)

plt.title(f"Comparación: Saturación real vs saturación BM11\npara el término '{term}'")
plt.xlabel("tf del término en cada documento")
plt.ylabel("Contribución normalizada")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 6. BM25: el estándar moderno

BM25 incorpora tres ideas clave:

1. Pesos RSJ probabilísticos.
2. Saturación del término (*k1*).
3. Normalización por longitud controlada por *b*.

$$
BM_{25}(D,Q)= \sum_{t \in Q}
w_t \cdot
\frac{tf_{t,D}(k_1+1)}
{tf_{t,D} + k_1\left(1 - b + b\frac{|D|}{avgdl}\right)}
$$

Valores típicos:
- $k_1\in[1.2,2.0]$
- $b\in[0.5,0.8]$

BM25 es hoy el estándar industrial y académico.

In [ ]:
def bm25(query, k1=1.5, b=0.75, topk=10):
    q = tokenize(query)
    avgdl = sum(doc_len) / len(doc_len)
    scores = []
    for i, tf in enumerate(doc_tf):
        dl = doc_len[i]
        s = 0
        for w in q:
            if w in df:
                wt = rsj(N, df, w)
                denom = tf[w] + k1 * (1 - b + b * (dl / avgdl))
                s += wt * ((tf[w] * (k1 + 1)) / denom)
        scores.append((s, docs[i]["id"]))
    return sorted(scores, reverse=True)[:topk]

### Visualizaciones BM25: efecto de k1 y b

In [ ]:
tf_vals = np.arange(1, 20)

k1 = 1.5
b = 0.75
dl = 150
avgdl = 100
bm25_tf = (tf_vals * (k1 + 1)) / (tf_vals + k1 * (1 - b + b * (dl / avgdl)))
plt.plot(tf_vals, bm25_tf)
plt.title("Saturación BM25")
plt.show()

In [ ]:
term = "glucose"
freqs = np.array([doc.count(term) for doc in tokenized_docs])

# tf > 0
tf_vals = np.sort(freqs[freqs > 0])

# ---- Longitudes ----
dl = np.array([len(doc) for doc in tokenized_docs])
dl_vals = dl[freqs > 0]
avgdl = dl.mean()

# ---- Parámetros BM25 ----
k1 = 1.2
b = 0.75

# ---- Fórmula BM25 TF completa ----
bm25_tf = (tf_vals * (k1 + 1)) / (tf_vals + k1 * (1 - b + b * (dl_vals / avgdl)))

# Normalizamos dividiendo por el máximo tf para visualizar
tf_real_norm = tf_vals / tf_vals.max()

# ---- Graficar ----
plt.figure(figsize=(8, 5))
plt.plot(tf_vals, tf_real_norm, "o-", label=f"Saturación REAL del término '{term}'")
plt.plot(tf_vals, bm25_tf, "s--", label="Saturación BM25", alpha=0.8)

plt.title(f"Comparación: Saturación real vs. saturación BM25\n(término: '{term}')")
plt.xlabel("tf del término en cada documento donde aparece")
plt.ylabel("Contribución normalizada")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 7. TF-IDF como referencia

In [ ]:
def tfidf(query, topk=10):
    q = tokenize(query)
    scores = []
    for i, tf in enumerate(doc_tf):
        s = 0
        for w in q:
            if w in tf:
                idf = math.log(N / (df[w] + 1))
                s += tf[w] * idf
        scores.append((s, docs[i]["id"]))
    return sorted(scores, reverse=True)[:topk]

## 8. Comparación final

In [ ]:
queries = [
    "cancer therapy",
    "heart disease",
    "brain tumor",
    "blood pressure",
]


def top10(scores):
    # scores es una lista de tuplas: (score, doc_id)
    indexed = []
    for s in scores:
        if isinstance(s, tuple):
            score = s[0]
            doc_id = s[1]
        else:
            # fallback, por si llegara algo extraño
            score = s
            doc_id = None
        indexed.append((doc_id, score))

    # ordenamos por score descendente
    indexed.sort(key=lambda x: x[1], reverse=True)
    return indexed[:10]


for q in queries:
    print("\n=====================================")
    print("Query:", q)
    print("=====================================")

    bm11_scores = bm11(q)
    bm25_scores = bm25(q)
    tfidf_scores = tfidf(q)

    print("\nTop-10 BM11:")
    for doc, score in top10(bm11_scores):
        print(f"Doc {doc} — Score {score:.4f}")

    print("\nTop-10 BM25:")
    for doc, score in top10(bm25_scores):
        print(f"Doc {doc} — Score {score:.4f}")

    print("\nTop-10 TF-IDF:")
    for doc, score in top10(tfidf_scores):
        print(f"Doc {doc} — Score {score:.4f}")

## 10. Autoevaluación y ejercicios

### 1. Explica por qué TF satura.

La saturación significa que un término aporta demasiada relevancia si aparece muchas veces en un documento. En el TF, este número es lineal, por eso se dice que TF satura porque la diferencia entre 1 y 2 apariciones es la misma que 1000 y 1001.


### 2. Compara BM11 y BM25.

In [ ]:
# For each query we are going to graph the order of all the documents
plt.figure(figsize=(8, 5))

# Number of documents to consider
topk = len(docs)

for q in queries:
    bm11Docs = [x[1] for x in bm11(q, topk=topk) if x[0] > 0]
    bm25Docs = [x[1] for x in bm25(q, topk=topk) if x[0] > 0]

    # Number of documents returned
    n = len(bm11Docs)

    x = list(range(n))
    y = list(bm25Docs.index(doc_id) for doc_id in bm11Docs)

    plt.plot(x, y, label=q)

    plt.xlabel("BM11 Rank")
    plt.ylabel("BM25 Rank")
    plt.title("BM11 vs BM25 Document Ranks")

plt.legend()
plt.show()

### 3. Ajusta b y analiza.

In [ ]:
term = "glucose"

bList = np.arange(0.0, 1.0, 0.01)

results = []

for b in bList:
    bm25Docs = bm25(q, topk=topk, b=b)

    # Eliminate documents with score 0
    bm25Docs = {doc[1] : doc[0] for doc in bm25Docs if doc[0] > 0}

    # Store results
    for doc, score in bm25Docs.items():
        results.append({"b": b, "Documento": doc, "Score_BM25": bm25Docs[doc]})

for doc in set(r["Documento"] for r in results):
    docResults = [r for r in results if r["Documento"] == doc]
    docResults = sorted(docResults, key=lambda x: x["b"])

    x = [r["b"] for r in docResults]
    y = [r["Score_BM25"] for r in docResults]

    plt.plot(x, y, label=f"Doc {doc}")

plt.xlabel("b")
plt.ylabel("Score BM25")
plt.title(f"BM25 Scores for term '{term}' across different b values")
plt.show()

### 4. Implementa BM25 sin normalización.

In [ ]:
def bm25NoNorm(query, k1=1.5, topk=10):
    # This is bm11
    return bm11(query, k=k1, topk=topk)

### 5. Analiza documentos largos vs cortos.

In [ ]:
plt.figure(figsize=(8, 5))

# Number of documents to consider
topk = len(docs)
term = "glucose"

bm11Docs = {x[1]: x[0] for x in bm11(term, topk=topk) if x[0] > 0}
bm25Docs = {x[1]: x[0] for x in bm25(term, topk=topk) if x[0] > 0}

# Order both by document length
docLengths = {d["id"]: len(tokenized_docs[d["id"]]) for d in docs}
bm11Docs = dict(sorted(bm11Docs.items(), key=lambda x: docLengths[x[0]]))
bm25Docs = dict(sorted(bm25Docs.items(), key=lambda x: docLengths[x[0]]))

# Plot
x = list(docLengths[doc_id] for doc_id in bm11Docs.keys())
y1 = list(bm11Docs[doc_id] for doc_id in bm11Docs.keys())
y2 = list(bm25Docs[doc_id] for doc_id in bm25Docs.keys())

plt.plot(x, y1, label="BM11")
plt.plot(x, y2, label="BM25")
plt.xlabel("Longitud del documento")
plt.ylabel("Score")
plt.title(f"BM11 vs BM25 Scores for term '{term}' by Document Length")
plt.legend()
plt.show()

# Ejercicios y cuestiones

## 1. Modifica el valor de *k* en BM11:

### ¿Cómo cambia el ranking?

### ¿Qué ocurre con términos repetidos muchas veces?

## 2. Cambia los valores de *k1* y *b* en BM25:

### ¿Qué valor parece más adecuado para MED.ALL?

## 3. Calcula manualmente el peso RSJ para un término frecuente y uno raro.

### ¿Por qué el raro aporta más evidencia?

## 4. Elimina la normalización por longitud en BM25 (pon *b=0*):

### Analiza qué ocurre con documentos largos.

## 5. Implementa una variante:

### BM25 sin saturación (lineal): compara resultados.

### ¿Por qué BM25 no es lineal en *tf*?

## 6. Busca una consulta propia del área biomédica y analiza:

### ¿Qué términos dominan los scores?

### ¿Qué documentos salen primeros?

### ¿Concuerda con tu percepción de relevancia?